# Constructing a RAG system using LlamaIndex and Ollama

This Jupyter notebook leverages Ollama and LlamaIndex, powered by Ryzen AI, to build a Retrieval-Augmented Generation (RAG) application. LlamaIndex facilitates the creation of a pipeline from reading PDFs to indexing datasets and building a query engine, while Lemonade server provides the backend service for large language model (LLM) inference.

https://rocm.docs.amd.com/projects/ai-developer-hub/en/latest/notebooks/inference/rag_ollama_llamaindex.html

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/ai-agents/rag/aup_config.py

In [ ]:
from aup_config import aup_setup
aup_setup()

## Build the RAG pipeline

This section explains how to configure and build the RAG pipeline.

### Set up indexing and the query engine
Import the necessary libraries:

In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
import sys
import logging

## Configure embedding and LLM models

LlamaIndex implements the Ollama client interface to interact with the Ollama service. In this example, it requests both embedding and LLM services from Ollama.

In [ ]:
# Set embedding model
emb_fn="nomic-embed-text"
Settings.embed_model = OllamaEmbedding(model_name=emb_fn)

# Set ollama model
Settings.llm = Ollama(model="llama3.1:8b", request_timeout=120.0)

## Download data for RAG

Download a PDF (for example, the Vitis HLS user guide) and save it

In [ ]:
import requests
import os

base_url = 'https://docs.amd.com/api/khub/maps/wsGrDyp6~9qclJFHVNa2XQ/attachments/U2wv_UklnZP1~CevZS_N4Q-wsGrDyp6~9qclJFHVNa2XQ/content?download=true&Ft-Calling-App=ft%2Fturnkey-portal&Ft-Calling-App-Version=5.1.22'
download_dir = 'data_hls'
pdf_filename = 'vitis_hls_ug.pdf'

In [ ]:
os.makedirs(download_dir, exist_ok=True)
if not os.path.isfile(os.path.join(download_dir, pdf_filename)):
    response = requests.get(base_url, stream=True)
    if response.status_code == 200:
        pdf_path = os.path.join(download_dir, pdf_filename)
        with open(pdf_path, 'wb') as file: # this triggers some activity in the NPU
            file.write(response.content)

In [ ]:
import pygit2
chls_path = os.path.join(download_dir, 'Vitis-HLS-Introductory-Examples')
if not os.path.exists(chls_path):
    pygit2.clone_repository("https://github.com/Xilinx/Vitis-HLS-Introductory-Examples", chls_path)

The SimpleDirectoryReader is the most commonly used data connector. Provide it with an input directory or a list of files and it selects the best file reader based on the file extensions.

In [ ]:
documents = SimpleDirectoryReader(input_dir=os.path.join(download_dir), recursive=False).load_data()
# Check the content
print(documents[10])

In [ ]:
print(f'{len(documents)=}')

In [ ]:
print(documents[599].text_resource.text)

## Create a vector dataset with Chroma

[Chroma DB](https://www.trychroma.com/) is a database that stores and queries embeddings, documents, and metadata for LLM apps that integrates well with LlamaIndex. It creates the vector dataset by sourcing the PDF file.

In [ ]:
# Initialize client and save data
db = chromadb.PersistentClient(path="./chroma_db_lemon/hls_db")
# create collection
chroma_collection = db.get_or_create_collection("hls_db")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
# Build vector index per-document
if chroma_collection.count() == 0:
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    vector_index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=20)],
    )
else:
    vector_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

## Create the query engine

Next, create the query engine with a response mode. Select the response mode based on your specific needs. For detailed guidance, see the [LlamaIndex response modes documentation](https://docs.llamaindex.ai/en/v0.10.19/module_guides/deploying/query_engine/response_modes.html).

In [ ]:
# Query your data
query_engine = vector_index.as_query_engine(response_mode="refine", similarity_top_k=5)

## Customize the query prompts

Updating Prompt for Q&A. Define task-specific prompts:

In [ ]:
from llama_index.core import PromptTemplate

template = (
    "You are a Vitis HLS product expert who is very familiar with the user guide and provides the guide to the end user.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the information from multiple sources and not prior knowledge\n"
    "answer the question according to the index dataset.\n"
    "if the question is not related to Vitis HLS and HLS, just say it is not related to my knowledge base.\n"
    "if you don't know the answer, just say that I don't know.\n"
    "Answers need to be precise and concise.\n"
    "if the question is in Chinese, please translate Chinese to English in advance"
    "Query: {query_str}\n"
    "Answer: "
)
qa_template = PromptTemplate(template)
query_engine.update_prompts(
    {"response_synthesizer:text_qa_template": qa_template}
)

template = (
    "The original query is as follows: {query_str}.\n"
    "We have provided an existing answer: {existing_answer}.\n"
    "We have the opportunity to refine the existing answer (only if needed) with some more context below.\n"
    "-------------\n"
    "{context_msg}\n"
    "-------------\n"
    "Given the new context, refine the original answer to better answer the query. If the context isn't useful, return the original answer.\n"
    "if the question is 'who are you', just say I am an expert on AMD Vitis HLS.\n"
    "Answers need to be precise and concise.\n"
    "Refined Answer: "
)

qa_template = PromptTemplate(template)

query_engine.update_prompts(
    {"response_synthesizer:refine_template": qa_template}
)

In [ ]:
response = query_engine.query("Who are you?")
print(response)

## Query examples

Run the following queries:

Query 1: Briefly describe the steps to install Vitis HLS?

In [ ]:
response = query_engine.query("is windows supported by Vitis HLS?")
print(response)

In [ ]:
print(response.source_nodes[0].node.text)

In [ ]:
len(response.source_nodes)

In [ ]:
print(response.source_nodes[0].node.metadata)

In [ ]:
print(response.source_nodes[2].score)

In [ ]:
response = query_engine.query("Where can I find tutorials and examples for Vitis HLS?")
print(response)

In [ ]:
response = query_engine.query("Provide the C/C++ code to write a vector add in Vitis HLS?")
print(response)

In [ ]:
response = query_engine.query("What does the dataflow pragma do?")
print(response)

In [ ]:
response = query_engine.query("What interfaces does Vitis HLS support?")
print(response)

In [ ]:
logging.basicConfig(stream=sys.stdout, level=logging.DEBUG, force=True)
response = query_engine.query("In what chapter can I find documentation about directives?")
print(response)

In [ ]:
query_engine.get_prompts()

## Conclusion

This tutorial demonstrates how to construct a RAG pipeline using LlamaIndex and Ollama on AMD Radeon GPUs with ROCm. For further details, see the documentation for the different components.

----------

Content curated by the AMD University Program team.

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.

SPDX-License-Identifier: MIT